##Entregable 1 (Baselines, auditoría de fuga de datos y embeddings)

1. Se verifica que el test de
   sub50_42 no se superpone con el train de full_42 (el split que
   preentrenó AstroM3-CLIP) por una posible fuga de datos.
2. Se carga el sub10_42 para el desarrollo local
3. Se entrenan los tres baselines unimodales: GRU para fotometría,
   CNN 1D para espectros y MLP para metadatos.
4. Se hace un experimento de generalización: modelos entrenados en sub10 y evaluados sobre el test de sub50, y luego entrenamiento formal sobre el train de sub50 y evaluado sobre su propio test.
5. Se extrae el embedding de la capa previa al clasificador de los modelos entrenados en sub50 para la fusión por concatenación.



In [ ]:
!pip install -q datasets torch scikit-learn

No es necesario iniciar sesión en Hugging Face, pero se lo hace para aumentar el límite de solicitudes y la velocidad de descarga.

In [ ]:
from huggingface_hub import login

login()  # pega aquí tu token cuando te lo pida (no queda guardado en el notebook)

Creamos el paquete src. Cada celda %%writefile escribe uno de los módulos necesarios para el programa

In [ ]:
import os
os.makedirs('src', exist_ok=True)

In [ ]:
%%writefile src/__init__.py
# -*- coding: utf-8 -*-


Writing src/__init__.py


Aquí manejo la configuración central. Definimos SEED = 42 en todo el proyecto para evitar la fuga de datos, según lo que documentan los autores de AstroM3.

In [ ]:
%%writefile src/config.py
# -*- coding: utf-8 -*-

SEED = 42
DATASET_NAME = "AstroMLCore/AstroM3Processed"

SUBSET_DEV = "sub10"      # desarrollo local
SUBSET_FORMAL = "sub50"   # entrenamiento formal
SUBSET_FULL = "full"      # para la detección de fuga de datos

NUM_CLASSES = 10
CLASS_NAMES = ["DSCT", "EA", "EB", "EW", "HADS", "M", "ROT", "RRAB", "RRC", "SR"]

#modelo preentrenado para la fusión por transfer learning contrastivo
CLIP_MODEL_NAME = "AstroMLCore/AstroM3-CLIP"

#misma dimensión de embedding para los tres codificadores unimodales (más conveniente para la fusión por concatenación)
EMBEDDING_DIM = 64

#hiperparámetros de entrenamiento
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
NUM_EPOCHS = 20
RANDOM_STATE_TORCH = SEED


Writing src/config.py


Aquí se hace la auditoría en caso de una posible fuga de datos. AstroM3-CLIP fue preentrenado sobre el split train del dataset full con seed 42. Aquí demuestro que ningún objeto del conjunto de prueba (split test de sub50_42) estuvo en ese train.

**Metodología**: cada estrella tiene un vector de metadatos catalográficos de 34
valores continuos (posición, magnitudes, periodo, amplitud, etc. — ver
AstroM3Processed en Hugging Face), en esencia, una huella
digital única por objeto. Si el mismo vector de metadatos aparece en full_42/train y en sub50_42/test, ese es el mismo objeto físico

In [ ]:
%%writefile src/audit.py
# -*- coding: utf-8 -*-

from dataclasses import dataclass, field
from typing import List, Optional, Sequence

from . import config


# n° de decimales al redondear los metadatos
FINGERPRINT_DECIMALS = 6


@dataclass
class LeakageAuditReport:
    n_train_full: int
    n_test_sub50: int
    n_overlap: int
    overlapping_ids: list = field(default_factory=list)

    @property
    def is_clean(self) -> bool:
        return self.n_overlap == 0

    def __str__(self):
        status = "SIN FUGA DETECTADA" if self.is_clean else "FUGA DETECTADA"
        lines = [
            "---- Auditoría de fuga de datos (AstroM3-CLIP) ----",
            f"Objetos en full/train:      {self.n_train_full}",
            f"Objetos en sub50/test:      {self.n_test_sub50}",
            f"Objetos en la intersección: {self.n_overlap}",
            f"Resultado: {status}",
        ]
        if not self.is_clean:
            preview = self.overlapping_ids[:5]
            lines.append(f"Ejemplos de objetos en fuga (máx 5): {preview}")
        return "\n".join(lines)


#hace al vector de metadata una tupla hasheable (el identificador del objeto)
def _metadata_fingerprint(metadata_row: Sequence[float],
                           decimals: int = FINGERPRINT_DECIMALS):
    return tuple(round(float(v), decimals) for v in metadata_row)


#calcula la huella de cada fila de un split
def _fingerprint_split(hf_split, metadata_key: str = "metadata",
                        decimals: int = FINGERPRINT_DECIMALS) -> List[tuple]:
    return [
        _metadata_fingerprint(row, decimals=decimals)
        for row in hf_split[metadata_key]
    ]

#ejecuta el audit
def run_leakage_audit(
    dataset_name: str = config.DATASET_NAME,
    seed: int = config.SEED
) -> LeakageAuditReport:

    from datasets import load_dataset

    full = load_dataset(dataset_name, name=f"full_{seed}")
    sub50 = load_dataset(dataset_name, name=f"sub50_{seed}")

    train_fps = _fingerprint_split(full["train"])
    test_fps = _fingerprint_split(sub50["test"])
    train_set, test_set = set(train_fps), set(test_fps)
    overlap = train_set & test_set

    return LeakageAuditReport(
        n_train_full=len(train_set),
        n_test_sub50=len(test_set),
        n_overlap=len(overlap),
        overlapping_ids=sorted(overlap)[:50],
    )


#chequeo de prueba entre splits train y test de sub10
def run_self_check(subset_name: str = "sub10", seed: int = config.SEED,
                    dataset_name: str = config.DATASET_NAME) -> LeakageAuditReport:

    from datasets import load_dataset

    small = load_dataset(dataset_name, name=f"{subset_name}_{seed}")

    train_fps = _fingerprint_split(small["train"])
    test_fps = _fingerprint_split(small["test"])
    train_set, test_set = set(train_fps), set(test_fps)
    overlap = train_set & test_set

    return LeakageAuditReport(
        n_train_full=len(train_set),
        n_test_sub50=len(test_set),
        n_overlap=len(overlap),
        overlapping_ids=sorted(overlap)[:50],
    )


#para ver las columnas disponibles y una fila de ejemplo
def inspect_available_columns(dataset_name: str = config.DATASET_NAME,
                               seed: int = config.SEED):

    from datasets import load_dataset

    full = load_dataset(dataset_name, name=f"full_{seed}")
    print("Columnas disponibles:", list(full["train"].features.keys()))
    print("\nPrimer ejemplo:")
    example = full["train"][0]
    for k, v in example.items():
        v_repr = type(v).__name__
        try:
            v_repr += f" (len={len(v)})"
        except TypeError:
            v_repr += f" = {v}"
        print(f"  {k}: {v_repr}")


Writing src/audit.py


Aquí empaquetamos los splits de Hugging Face (por requerimiento de PyTorch). Se implementa collate_fn porque la fotometría de cada estrella tiene longitud distinta. Se resuelve aplicando un padding.

In [ ]:
%%writefile src/datasets.py
# -*- coding: utf-8 -*-

from typing import Dict, List

import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader

from . import config

PHOTOMETRY_KEY = "photometry"
SPECTRA_KEY = "spectra"
METADATA_KEY = "metadata"
LABEL_KEY = "label"


#verifica formato del dataset
def inspect_sample(hf_split) -> None:
    example = hf_split[0]
    print("Claves disponibles:", list(example.keys()))
    for key, value in example.items():
        try:
            arr = torch.as_tensor(value)
            print(f"  {key}: tensor de forma {tuple(arr.shape)}, dtype origen {type(value)}")
        except (TypeError, ValueError):
            print(f"  {key}: {value!r} (no numérico / no convertible directo)")


#envuelve un split de la BD AstroM3Processed
class AstroMultimodalDataset(Dataset):

    def __init__(self, hf_split, photometry_key=PHOTOMETRY_KEY,
                 spectra_key=SPECTRA_KEY, metadata_key=METADATA_KEY,
                 label_key=LABEL_KEY):
        self.data = hf_split
        self.photometry_key = photometry_key
        self.spectra_key = spectra_key
        self.metadata_key = metadata_key
        self.label_key = label_key

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        row = self.data[idx]

        photometry = torch.as_tensor(row[self.photometry_key], dtype=torch.float32)
        #forma (T, canales)
        if photometry.ndim == 1:
            photometry = photometry.unsqueeze(-1)

        spectra = torch.as_tensor(row[self.spectra_key], dtype=torch.float32)

        metadata = torch.as_tensor(row[self.metadata_key], dtype=torch.float32).flatten()

        label = int(row[self.label_key])

        return {
            "photometry": photometry,
            "spectra": spectra,
            "metadata": metadata,
            "label": label,
        }


#arma un lote rellenando la fotometria a la longitud máx. del batch. Devuelvo la longitud real para poder empaquetar la secuencia en el GRU.
def collate_multimodal(batch: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:

    photometry_list = [b["photometry"] for b in batch]
    lengths = torch.tensor([p.shape[0] for p in photometry_list], dtype=torch.long)
    photometry_padded = pad_sequence(photometry_list, batch_first=True)  # (B, T_max, C)

    spectra = torch.stack([b["spectra"] for b in batch])      # (B, 3, L)
    metadata = torch.stack([b["metadata"] for b in batch])    # (B, D)
    labels = torch.tensor([b["label"] for b in batch], dtype=torch.long)

    return {
        "photometry": photometry_padded,
        "photometry_lengths": lengths,
        "spectra": spectra,
        "metadata": metadata,
        "label": labels,
    }


def make_dataloader(hf_split, batch_size: int = config.BATCH_SIZE,
                     shuffle: bool = True, **dataset_kwargs) -> DataLoader:
    ds = AstroMultimodalDataset(hf_split, **dataset_kwargs)
    return DataLoader(
        ds, batch_size=batch_size, shuffle=shuffle,
        collate_fn=collate_multimodal,
    )

#carga un subset con la semilla definida
def load_subset(subset_name: str, seed: int = config.SEED,
                 dataset_name: str = config.DATASET_NAME):
    from datasets import load_dataset
    return load_dataset(dataset_name, name=f"{subset_name}_{seed}")


Writing src/datasets.py


Aquí se implementan los tres codificadores unimodales de referencia (GRU - fotometría, CNN 1D - espectro, MLP - metadata). Los tres devuelven un embedding de la misma dimensión para poder hacer la fusión por concatenación después

In [ ]:
%%writefile src/models.py
# -*- coding: utf-8 -*-

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence

from . import config


#GRU sobre la curva de luz. Recibe la secuencia con el padding y el vector con la longitud real para ignorar el relleno en el entrenamiento.
class PhotometryGRUEncoder(nn.Module):

    def __init__(self, input_dim: int = 9, hidden_dim: int = 64,
                 num_layers: int = 1, embedding_dim: int = config.EMBEDDING_DIM,
                 dropout: float = 0.1):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_dim, hidden_size=hidden_dim,
            num_layers=num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.proj = nn.Linear(hidden_dim, embedding_dim)
        self.norm = nn.LayerNorm(embedding_dim)

    def forward(self, photometry: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        # photometry: (B, T_max, input_dim); lengths: (B,)
        packed = pack_padded_sequence(
            photometry, lengths.cpu(), batch_first=True, enforce_sorted=False,
        )
        _, h_n = self.gru(packed)
        last_layer_hidden = h_n[-1]                # (B, hidden_dim)
        embedding = self.norm(self.proj(last_layer_hidden))
        return embedding                            # (B, embedding_dim)


#CNN 1D para espectro.
class SpectraCNNEncoder(nn.Module):

    def __init__(self, in_channels: int = 3, embedding_dim: int = config.EMBEDDING_DIM,
                 dropout: float = 0.1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(4),

            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(4),

            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool1d(1),
        )
        self.dropout = nn.Dropout(dropout)
        self.proj = nn.Linear(64, embedding_dim)
        self.norm = nn.LayerNorm(embedding_dim)

    def forward(self, spectra: torch.Tensor) -> torch.Tensor:
        features = self.conv(spectra).squeeze(-1)
        features = self.dropout(features)
        embedding = self.norm(self.proj(features))
        return embedding


#MLP para metadata
class MetadataMLPEncoder(nn.Module):

    def __init__(self, input_dim: int, hidden_dim: int = 64,
                 embedding_dim: int = config.EMBEDDING_DIM, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(inplace=True),
        )
        self.proj = nn.Linear(hidden_dim, embedding_dim)
        self.norm = nn.LayerNorm(embedding_dim)

    def forward(self, metadata: torch.Tensor) -> torch.Tensor:
        features = self.net(metadata)
        embedding = self.norm(self.proj(features))
        return embedding

#design pattern que envuelve todos los encoders igual. Devuelve los logits y el embedding
class UnimodalClassifier(nn.Module):

    def __init__(self, encoder: nn.Module, embedding_dim: int = config.EMBEDDING_DIM,
                 num_classes: int = config.NUM_CLASSES):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(embedding_dim, num_classes)

    def forward(self, *args, **kwargs):
        embedding = self.encoder(*args, **kwargs)
        logits = self.head(embedding)
        return logits, embedding


Writing src/models.py


Loop de entrenamiento y evaluación que usan los tres baselines. Reporta exactitud. macroF1, precisión, recall y matriz de confusión

In [ ]:
%%writefile src/engine.py
# -*- coding: utf-8 -*-

from dataclasses import dataclass
from typing import Callable, Dict, List, Optional

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
                              confusion_matrix, classification_report)

from . import config


@dataclass
class EpochResult:
    loss: float
    accuracy: float
    macro_f1: float


def _move_batch_to_device(batch: Dict[str, torch.Tensor], device: str) -> Dict[str, torch.Tensor]:
    return {k: (v.to(device) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}


#para que el loop sea el mismo para las tres baselines
def train_one_epoch(model: nn.Module, dataloader, optimizer, device: str,
                     forward_fn: Callable) -> EpochResult:
    model.train()
    criterion = nn.CrossEntropyLoss()
    total_loss, all_preds, all_labels = 0.0, [], []

    for batch in dataloader:
        batch = _move_batch_to_device(batch, device)
        optimizer.zero_grad()
        logits, labels = forward_fn(model, batch)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        all_preds.append(logits.argmax(dim=1).detach().cpu().numpy())
        all_labels.append(labels.detach().cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    return EpochResult(
        loss=total_loss / len(all_labels),
        accuracy=accuracy_score(all_labels, all_preds),
        macro_f1=f1_score(all_labels, all_preds, average="macro", zero_division=0),
    )


#exactitud, macro-F1, precisión/recall por clase y matriz de confusión
@torch.no_grad()
def evaluate(model: nn.Module, dataloader, device: str, forward_fn: Callable,
             class_names: Optional[List[str]] = None) -> Dict:
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss, all_preds, all_labels = 0.0, [], []

    for batch in dataloader:
        batch = _move_batch_to_device(batch, device)
        logits, labels = forward_fn(model, batch)
        loss = criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)
        all_preds.append(logits.argmax(dim=1).cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    precision, recall, f1, support = precision_recall_fscore_support(
        all_labels, all_preds, average=None, zero_division=0,
        labels=list(range(config.NUM_CLASSES)),
    )
    names = class_names or [str(i) for i in range(config.NUM_CLASSES)]

    return {
        "loss": total_loss / len(all_labels),
        "accuracy": accuracy_score(all_labels, all_preds),
        "macro_f1": f1_score(all_labels, all_preds, average="macro", zero_division=0),
        "per_class": {
            names[i]: {"precision": precision[i], "recall": recall[i],
                       "f1": f1[i], "support": int(support[i])}
            for i in range(len(names))
        },
        "confusion_matrix": confusion_matrix(
            all_labels, all_preds, labels=list(range(config.NUM_CLASSES))),
        "y_true": all_labels,
        "y_pred": all_preds,
        "report_text": classification_report(
            all_labels, all_preds, labels=list(range(config.NUM_CLASSES)),
            target_names=names, zero_division=0),
    }

#entrena epochs y evalúa en validation tras cada una. Devuelve historial y métricas finales
def fit(model: nn.Module, train_loader, val_loader, forward_fn: Callable,
        device: str = "cpu", num_epochs: int = config.NUM_EPOCHS,
        lr: float = config.LEARNING_RATE, class_names: Optional[List[str]] = None,
        verbose: bool = True) -> Dict:
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []

    for epoch in range(1, num_epochs + 1):
        train_result = train_one_epoch(model, train_loader, optimizer, device, forward_fn)
        val_metrics = evaluate(model, val_loader, device, forward_fn, class_names)
        history.append({
            "epoch": epoch,
            "train_loss": train_result.loss, "train_acc": train_result.accuracy,
            "train_macro_f1": train_result.macro_f1,
            "val_loss": val_metrics["loss"], "val_acc": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
        })
        if verbose:
            print(f"[{epoch:02d}/{num_epochs}] "
                  f"train_loss={train_result.loss:.4f} train_acc={train_result.accuracy:.3f} | "
                  f"val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['accuracy']:.3f} "
                  f"val_macroF1={val_metrics['macro_f1']:.3f}")

    return {"history": history, "final_val_metrics": val_metrics}


#forward_fn por modalidad

def forward_fn_photometry(model, batch):
    logits, _ = model(batch["photometry"], batch["photometry_lengths"])
    return logits, batch["label"]


def forward_fn_spectra(model, batch):
    logits, _ = model(batch["spectra"])
    return logits, batch["label"]


def forward_fn_metadata(model, batch):
    logits, _ = model(batch["metadata"])
    return logits, batch["label"]


Writing src/engine.py


Aquí se extraen los embeddings de cada modelo ya entrenado. Se guardan para poder concatenarlos después

In [ ]:
%%writefile src/embeddings.py
# -*- coding: utf-8 -*-

from typing import Dict

import numpy as np
import torch
import torch.nn as nn

from . import engine

#corre el codificador sobre todo un split y devuelve embeddings + etiquetas como arrays
@torch.no_grad()
def extract_embeddings(model: nn.Module, dataloader, device: str,
                        forward_fn) -> Dict[str, np.ndarray]:
    model.eval()
    all_embeddings, all_labels = [], []

    for batch in dataloader:
        batch = engine._move_batch_to_device(batch, device)
        _, labels = forward_fn_labels_only(batch)
        embedding = _forward_embedding_only(model, batch, forward_fn)
        all_embeddings.append(embedding.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    return {
        "embeddings": np.concatenate(all_embeddings, axis=0),
        "labels": np.concatenate(all_labels, axis=0),
    }


def forward_fn_labels_only(batch):
    return None, batch["label"]


#reutiliza la lógica de forward_fn pero para los embeddings
def _forward_embedding_only(model, batch, forward_fn):
    if forward_fn is engine.forward_fn_photometry:
        _, embedding = model(batch["photometry"], batch["photometry_lengths"])
    elif forward_fn is engine.forward_fn_spectra:
        _, embedding = model(batch["spectra"])
    elif forward_fn is engine.forward_fn_metadata:
        _, embedding = model(batch["metadata"])
    else:
        raise ValueError("forward_fn no reconocido")
    return embedding


def save_embeddings(path: str, embeddings: np.ndarray, labels: np.ndarray) -> None:
    np.savez(path, embeddings=embeddings, labels=labels)
    print(f"Guardado: {path}  (embeddings: {embeddings.shape}, labels: {labels.shape})")


def load_embeddings(path: str) -> Dict[str, np.ndarray]:
    data = np.load(path)
    return {"embeddings": data["embeddings"], "labels": data["labels"]}


#concatena los tres embeddings
def concatenate_modalities(photometry_emb: np.ndarray, spectra_emb: np.ndarray,
                            metadata_emb: np.ndarray, labels: np.ndarray) -> Dict[str, np.ndarray]:
    assert photometry_emb.shape[0] == spectra_emb.shape[0] == metadata_emb.shape[0], \
        "Las tres modalidades deben tener el mismo número de ejemplos y en el mismo orden."
    fused = np.concatenate([photometry_emb, spectra_emb, metadata_emb], axis=1)
    return {"embeddings": fused, "labels": labels}


Writing src/embeddings.py


In [ ]:
# reiniciar el intérprete de imports (por si ya se había importado src antes)
import importlib, sys
for m in list(sys.modules):
    if m == "src" or m.startswith("src."):
        del sys.modules[m]

from src import config, audit, engine, embeddings as emb_module
from src.datasets import load_subset, inspect_sample, make_dataloader, AstroMultimodalDataset
from src.models import PhotometryGRUEncoder, SpectraCNNEncoder, MetadataMLPEncoder, UnimodalClassifier
import torch

torch.manual_seed(config.SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", device)

Dispositivo: cpu


**Auditoría de fuga de datos**

Se verifica que el test de sub50_42 no se superpone con
el train de full_42.


In [ ]:
# prueba que la auditoria funciona en sub10
self_check = audit.run_self_check(subset_name="sub10")
print(self_check)
assert self_check.is_clean, (
    "El self-check encontró fuga entre su propio "
    "train y test. chequear el método de auditoria"
)

print()

#la auditoría real con full_42
report = audit.run_leakage_audit()
print(report)
assert report.is_clean, "Hay fuga de datos"

README.md:   0%|          | 0.00/23.5k [00:00<?, ?B/s]

sub10_42/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 45.2MB            

sub10_42/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sub10_42/validation-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B / 6.26MB            

sub10_42/validation-00000-of-00001.parqu(…): downloading bytes:           |  0.00B            

...

sub50_42/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/8526 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1071 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1111 [00:00<?, ? examples/s]

---- Auditoría de fuga de datos (AstroM3-CLIP) ----
Objetos en full/train:      17063
Objetos en sub50/test:      1111
Objetos en la intersección: 0
Resultado: SIN FUGA DETECTADA


Carga de sub10_42



In [ ]:
sub10 = load_subset("sub10")
print(sub10)
print()
inspect_sample(sub10["train"])

DatasetDict({
    train: Dataset({
        features: ['photometry', 'spectra', 'metadata', 'label'],
        num_rows: 1660
    })
    validation: Dataset({
        features: ['photometry', 'spectra', 'metadata', 'label'],
        num_rows: 210
    })
    test: Dataset({
        features: ['photometry', 'spectra', 'metadata', 'label'],
        num_rows: 220
    })
})

Claves disponibles: ['photometry', 'spectra', 'metadata', 'label']
  photometry: tensor de forma (192, 9), dtype origen <class 'list'>
  spectra: tensor de forma (3, 2575), dtype origen <class 'list'>
  metadata: tensor de forma (34,), dtype origen <class 'list'>
  label: tensor de forma (), dtype origen <class 'int'>


In [ ]:
import src.datasets as ds_module

#nombre de las clases en la DB
real_class_names = sub10["train"].features["label"].names
print("Clases en el dataset:", real_class_names)
assert real_class_names == config.CLASS_NAMES, "Ajusta config.CLASS_NAMES al orden real antes de seguir."

# dimensión real de metadata (para construir el MLP)
metadata_dim = len(sub10["train"][0][ds_module.METADATA_KEY])
print("Dimensión de metadatos:", metadata_dim)

Clases en el dataset: ['DSCT', 'EA', 'EB', 'EW', 'HADS', 'M', 'ROT', 'RRAB', 'RRC', 'SR']
Dimensión de metadatos: 34


**Entrenar los tres baselines sobre sub10**

Aquí se entrenan las tres modalidades, empezando por metadatos, luego espectros, y al final fotometría


In [ ]:
train_loader = make_dataloader(sub10["train"], batch_size=config.BATCH_SIZE, shuffle=True)
val_loader   = make_dataloader(sub10["validation"], batch_size=config.BATCH_SIZE, shuffle=False)
test_loader  = make_dataloader(sub10["test"], batch_size=config.BATCH_SIZE, shuffle=False)

In [ ]:
# --- Metadatos (MLP) ---
mlp_model = UnimodalClassifier(MetadataMLPEncoder(input_dim=metadata_dim), num_classes=config.NUM_CLASSES)
mlp_result = engine.fit(mlp_model, train_loader, val_loader,
                         forward_fn=engine.forward_fn_metadata, device=device,
                         class_names=config.CLASS_NAMES)

[01/20] train_loss=1.6352 train_acc=0.414 | val_loss=1.2022 val_acc=0.581 val_macroF1=0.544
[02/20] train_loss=1.0751 train_acc=0.593 | val_loss=0.9956 val_acc=0.667 val_macroF1=0.661
[03/20] train_loss=0.9243 train_acc=0.651 | val_loss=0.9569 val_acc=0.662 val_macroF1=0.651
[04/20] train_loss=0.8427 train_acc=0.690 | val_loss=0.8988 val_acc=0.700 val_macroF1=0.679
[05/20] train_loss=0.7916 train_acc=0.705 | val_loss=0.8763 val_acc=0.695 val_macroF1=0.686
[06/20] train_loss=0.7669 train_acc=0.702 | val_loss=0.8492 val_acc=0.700 val_macroF1=0.679
[07/20] train_loss=0.7219 train_acc=0.734 | val_loss=0.8537 val_acc=0.667 val_macroF1=0.657
[08/20] train_loss=0.6933 train_acc=0.743 | val_loss=0.8167 val_acc=0.738 val_macroF1=0.728
[09/20] train_loss=0.6636 train_acc=0.747 | val_loss=0.8084 val_acc=0.719 val_macroF1=0.709
[10/20] train_loss=0.6575 train_acc=0.756 | val_loss=0.8088 val_acc=0.714 val_macroF1=0.709
[11/20] train_loss=0.6375 train_acc=0.759 | val_loss=0.7843 val_acc=0.719 val_ma

In [ ]:
# --- Espectros (CNN 1D) ---
cnn_model = UnimodalClassifier(SpectraCNNEncoder(in_channels=3), num_classes=config.NUM_CLASSES)
cnn_result = engine.fit(cnn_model, train_loader, val_loader,
                         forward_fn=engine.forward_fn_spectra, device=device,
                         class_names=config.CLASS_NAMES)

[01/20] train_loss=1.9749 train_acc=0.269 | val_loss=1.8675 val_acc=0.310 val_macroF1=0.268
[02/20] train_loss=1.7936 train_acc=0.329 | val_loss=1.7794 val_acc=0.386 val_macroF1=0.330
[03/20] train_loss=1.7359 train_acc=0.369 | val_loss=1.7942 val_acc=0.362 val_macroF1=0.299
[04/20] train_loss=1.6830 train_acc=0.369 | val_loss=1.6834 val_acc=0.352 val_macroF1=0.315
[05/20] train_loss=1.6372 train_acc=0.390 | val_loss=1.7703 val_acc=0.343 val_macroF1=0.303
[06/20] train_loss=1.6144 train_acc=0.396 | val_loss=1.5527 val_acc=0.438 val_macroF1=0.385
[07/20] train_loss=1.5430 train_acc=0.420 | val_loss=1.6967 val_acc=0.386 val_macroF1=0.324
[08/20] train_loss=1.5573 train_acc=0.420 | val_loss=1.5093 val_acc=0.457 val_macroF1=0.399
[09/20] train_loss=1.5039 train_acc=0.434 | val_loss=1.5182 val_acc=0.462 val_macroF1=0.442
[10/20] train_loss=1.4891 train_acc=0.425 | val_loss=1.5833 val_acc=0.405 val_macroF1=0.370
[11/20] train_loss=1.5019 train_acc=0.423 | val_loss=1.4752 val_acc=0.476 val_ma

In [ ]:
# --- Fotometría (GRU) ---
gru_model = UnimodalClassifier(PhotometryGRUEncoder(input_dim=9), num_classes=config.NUM_CLASSES)
gru_result = engine.fit(gru_model, train_loader, val_loader,
                         forward_fn=engine.forward_fn_photometry, device=device,
                         class_names=config.CLASS_NAMES)

[01/20] train_loss=1.7411 train_acc=0.395 | val_loss=1.2358 val_acc=0.614 val_macroF1=0.578
[02/20] train_loss=1.0257 train_acc=0.662 | val_loss=0.8657 val_acc=0.733 val_macroF1=0.720
[03/20] train_loss=0.7653 train_acc=0.753 | val_loss=0.6754 val_acc=0.786 val_macroF1=0.780
[04/20] train_loss=0.6398 train_acc=0.770 | val_loss=0.5933 val_acc=0.786 val_macroF1=0.775
[05/20] train_loss=0.5539 train_acc=0.801 | val_loss=0.5583 val_acc=0.795 val_macroF1=0.786
[06/20] train_loss=0.5162 train_acc=0.809 | val_loss=0.5082 val_acc=0.762 val_macroF1=0.756
[07/20] train_loss=0.5145 train_acc=0.807 | val_loss=0.5203 val_acc=0.795 val_macroF1=0.785
[08/20] train_loss=0.4896 train_acc=0.802 | val_loss=0.4955 val_acc=0.795 val_macroF1=0.793
[09/20] train_loss=0.4532 train_acc=0.827 | val_loss=0.4376 val_acc=0.833 val_macroF1=0.828
[10/20] train_loss=0.4395 train_acc=0.826 | val_loss=0.4274 val_acc=0.795 val_macroF1=0.792
[11/20] train_loss=0.4302 train_acc=0.831 | val_loss=0.4451 val_acc=0.814 val_ma

In [ ]:
# metricas finales de los tres baselines sobre el test de sub10
for name, model, forward_fn in [("Metadatos (MLP)", mlp_model, engine.forward_fn_metadata),
                                 ("Espectros (CNN 1D)", cnn_model, engine.forward_fn_spectra),
                                 ("Fotometría (GRU)", gru_model, engine.forward_fn_photometry)]:
    m = engine.evaluate(model, test_loader, device, forward_fn, config.CLASS_NAMES)
    print(f"\n=== {name} — test de sub10 ===")
    print(f"Accuracy: {m['accuracy']:.3f}  |  Macro-F1: {m['macro_f1']:.3f}")
    print(m["report_text"])


=== Metadatos (MLP) — test de sub10 ===
Accuracy: 0.750  |  Macro-F1: 0.734
              precision    recall  f1-score   support

        DSCT       0.58      0.82      0.68        22
          EA       0.67      0.36      0.47        22
          EB       0.56      0.23      0.32        22
          EW       0.57      0.77      0.65        22
        HADS       0.72      0.82      0.77        22
           M       0.91      0.95      0.93        22
         ROT       0.83      0.91      0.87        22
        RRAB       0.95      0.86      0.90        22
         RRC       0.76      0.86      0.81        22
          SR       0.95      0.91      0.93        22

    accuracy                           0.75       220
   macro avg       0.75      0.75      0.73       220
weighted avg       0.75      0.75      0.73       220


=== Espectros (CNN 1D) — test de sub10 ===
Accuracy: 0.491  |  Macro-F1: 0.453
              precision    recall  f1-score   support

        DSCT       0.34      

**Experimento de generalización**

1.   modelos entrenados en sub10, evaluados sobre el test de sub50
2.   entrenamiento sobre el train de sub50, evaluado sobre su propio test

In [ ]:
sub50 = load_subset("sub50")
sub50_test_loader = make_dataloader(sub50["test"], batch_size=config.BATCH_SIZE, shuffle=False)

print("---- Modelos entrenados en sub10, evaluados en test de sub50 ----")
for name, model, forward_fn in [("Metadatos (MLP)", mlp_model, engine.forward_fn_metadata),
                                 ("Espectros (CNN 1D)", cnn_model, engine.forward_fn_spectra),
                                 ("Fotometría (GRU)", gru_model, engine.forward_fn_photometry)]:
    m = engine.evaluate(model, sub50_test_loader, device, forward_fn, config.CLASS_NAMES)
    print(f"{name}: accuracy={m['accuracy']:.3f}  macro-F1={m['macro_f1']:.3f}")

---- Modelos entrenados en sub10, evaluados en test de sub50 ----
Metadatos (MLP): accuracy=0.683  macro-F1=0.630
Espectros (CNN 1D): accuracy=0.550  macro-F1=0.458
Fotometría (GRU): accuracy=0.663  macro-F1=0.663


In [ ]:
#entrenamiento sobre sub50 train, validado sobre sub50 validation
sub50_train_loader = make_dataloader(sub50["train"], batch_size=config.BATCH_SIZE, shuffle=True)
sub50_val_loader   = make_dataloader(sub50["validation"], batch_size=config.BATCH_SIZE, shuffle=False)

mlp_model_formal = UnimodalClassifier(MetadataMLPEncoder(input_dim=metadata_dim), num_classes=config.NUM_CLASSES)
mlp_formal_result = engine.fit(mlp_model_formal, sub50_train_loader, sub50_val_loader,
                                forward_fn=engine.forward_fn_metadata, device=device,
                                class_names=config.CLASS_NAMES)

cnn_model_formal = UnimodalClassifier(SpectraCNNEncoder(in_channels=3), num_classes=config.NUM_CLASSES)
cnn_formal_result = engine.fit(cnn_model_formal, sub50_train_loader, sub50_val_loader,
                                forward_fn=engine.forward_fn_spectra, device=device,
                                class_names=config.CLASS_NAMES)

gru_model_formal = UnimodalClassifier(PhotometryGRUEncoder(input_dim=9), num_classes=config.NUM_CLASSES)
gru_formal_result = engine.fit(gru_model_formal, sub50_train_loader, sub50_val_loader,
                                forward_fn=engine.forward_fn_photometry, device=device,
                                class_names=config.CLASS_NAMES)

[01/20] train_loss=0.9476 train_acc=0.663 | val_loss=0.7427 val_acc=0.727 val_macroF1=0.595
[02/20] train_loss=0.7098 train_acc=0.735 | val_loss=0.6969 val_acc=0.758 val_macroF1=0.695
[03/20] train_loss=0.6607 train_acc=0.750 | val_loss=0.6490 val_acc=0.759 val_macroF1=0.671
[04/20] train_loss=0.6305 train_acc=0.758 | val_loss=0.6155 val_acc=0.785 val_macroF1=0.751
[05/20] train_loss=0.6099 train_acc=0.769 | val_loss=0.6176 val_acc=0.780 val_macroF1=0.725
[06/20] train_loss=0.5899 train_acc=0.781 | val_loss=0.5952 val_acc=0.790 val_macroF1=0.770
[07/20] train_loss=0.5780 train_acc=0.781 | val_loss=0.5878 val_acc=0.784 val_macroF1=0.751
[08/20] train_loss=0.5532 train_acc=0.791 | val_loss=0.5787 val_acc=0.795 val_macroF1=0.775
[09/20] train_loss=0.5445 train_acc=0.795 | val_loss=0.6069 val_acc=0.786 val_macroF1=0.755
[10/20] train_loss=0.5338 train_acc=0.799 | val_loss=0.5570 val_acc=0.807 val_macroF1=0.793
[11/20] train_loss=0.5237 train_acc=0.804 | val_loss=0.5639 val_acc=0.800 val_ma

In [ ]:
print("---- Entrenamiento en sub50, evaluados en su propio test ----")
for name, model, forward_fn in [("Metadatos (MLP)", mlp_model_formal, engine.forward_fn_metadata),
                                 ("Espectros (CNN 1D)", cnn_model_formal, engine.forward_fn_spectra),
                                 ("Fotometría (GRU)", gru_model_formal, engine.forward_fn_photometry)]:
    m = engine.evaluate(model, sub50_test_loader, device, forward_fn, config.CLASS_NAMES)
    print(f"{name}: accuracy={m['accuracy']:.3f}  macro-F1={m['macro_f1']:.3f}")

---- Entrenamiento en sub50, evaluados en su propio test ----
Metadatos (MLP): accuracy=0.820  macro-F1=0.788
Espectros (CNN 1D): accuracy=0.609  macro-F1=0.478
Fotometría (GRU): accuracy=0.886  macro-F1=0.897


**Extracción de embeddings (para la fusión por concatenación)**

Aquí se extrae el embedding de la capa previa al clasificador de los modelos
entrenados en sub50 sobre los 3 splits


In [ ]:
loaders_ordered = {
    "train": make_dataloader(sub50["train"], batch_size=config.BATCH_SIZE, shuffle=False),
    "validation": make_dataloader(sub50["validation"], batch_size=config.BATCH_SIZE, shuffle=False),
    "test": sub50_test_loader,
}

os.makedirs("embeddings", exist_ok=True)

for split_name, loader in loaders_ordered.items():
    mlp_out = emb_module.extract_embeddings(mlp_model_formal, loader, device, engine.forward_fn_metadata)
    cnn_out = emb_module.extract_embeddings(cnn_model_formal, loader, device, engine.forward_fn_spectra)
    gru_out = emb_module.extract_embeddings(gru_model_formal, loader, device, engine.forward_fn_photometry)

    assert (mlp_out["labels"] == cnn_out["labels"]).all() and (mlp_out["labels"] == gru_out["labels"]).all(), \
        "labels no coinciden entre modalidades"

    emb_module.save_embeddings(f"embeddings/metadata_{split_name}.npz", mlp_out["embeddings"], mlp_out["labels"])
    emb_module.save_embeddings(f"embeddings/spectra_{split_name}.npz", cnn_out["embeddings"], cnn_out["labels"])
    emb_module.save_embeddings(f"embeddings/photometry_{split_name}.npz", gru_out["embeddings"], gru_out["labels"])

    fused = emb_module.concatenate_modalities(
        gru_out["embeddings"], cnn_out["embeddings"], mlp_out["embeddings"], mlp_out["labels"])
    emb_module.save_embeddings(f"embeddings/fused_{split_name}.npz", fused["embeddings"], fused["labels"])
